# Reproducing results on the Sage paper

## Packages

In [1]:
! pip uninstall sage -y
! pip install -e ../

Found existing installation: sage 0.0.1
Uninstalling sage-0.0.1:
  Successfully uninstalled sage-0.0.1
Obtaining file:///home/nnarenraju/Research/sage
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for sage (pyproject.toml) ... done
  Created wheel for sage: filename=sage-0.0.1-0.editable-py3-none-any.whl size=3623 sha256=917750072ad6f89653269cc69124546951e3483ff8c28f17d7f4f0a34f1aeabc
  Stored in directory: /tmp/pip-ephem-wheel-cache-nxcqzp4m/wheels/9b/47/cf/bfd99e705866b88810d9db84a85969530a2783f51228948a36
Successfully built sage


## Generating/downloading necessary data

In [2]:
from sage.data.download import get_segments as gs

In [3]:
print(gs.get_all_detnames())
print(gs.get_all_runnames())

{'H1', 'K1', 'L1', 'H2', 'G1', 'V1'}
['O1', 'O2', 'O3GK', 'O3a', 'O3b', 'O4a', 'O4b1Disc', 'O4b2Disc', 'O4b3Disc', 'S5', 'S6']


In [4]:
tq = gs.TimelineQuery(detector=["H1", "L1", "V1"], 
                      observing_run=["O3a",],
                      start = 1238166018,
                      end = 1238176018,
                      auto_clean_empty_timelines=True)
                                
tq.download_segments()

2025-12-13 19:14:36 | INFO     | sage.data.download.get_segments:287 | Getting all segments between 1238166018-1238176018 for all detectors in the correct observing run


In [5]:
tq.timeline

array([('H1', 'H1_DATA', 1.23816602e+09, 1.23817602e+09, 'O3a', array([[1238166018, 1238170549],
              [1238170954, 1238172929],
              [1238172987, 1238176018]]))                                                       ,
       ('L1', 'L1_DATA', 1.23816602e+09, 1.23817602e+09, 'O3a', array([[1238166018, 1238170289],
              [1238175433, 1238176018]]))                                                       ,
       ('V1', 'V1_DATA', 1.23816602e+09, 1.23817602e+09, 'O3a', array([[1238166018, 1238176018]]))],
      dtype=[('detector', '<U2'), ('flag', '<U10'), ('start_time', '<f8'), ('end_time', '<f8'), ('observing_run', '<U10'), ('segments', 'O')])

In [6]:
tq.prune_segments(
    rm_short_segments = True,
    rm_min_duration = 22.0,
    rm_allevents = False,
    rm_window_length = 30,
)

2025-12-13 19:14:38 | INFO     | sage.data.download.get_segments:550 | Removing segments below the min duration of 22.0


In [7]:
tq.timeline

array([('H1', 'H1_DATA', 1.23816602e+09, 1.23817602e+09, 'O3a', array([[1238166018, 1238170549],
              [1238170954, 1238172929],
              [1238172987, 1238176018]]))                                                       ,
       ('L1', 'L1_DATA', 1.23816602e+09, 1.23817602e+09, 'O3a', array([[1238166018, 1238170289],
              [1238175433, 1238176018]]))                                                       ,
       ('V1', 'V1_DATA', 1.23816602e+09, 1.23817602e+09, 'O3a', array([[1238166018, 1238176018]]))],
      dtype=[('detector', '<U2'), ('flag', '<U10'), ('start_time', '<f8'), ('end_time', '<f8'), ('observing_run', '<U10'), ('segments', 'O')])

In [8]:
from sage.data.download import get_data_release

In [9]:
drd = get_data_release.DataReleaseDownloader(
    segments_metadata=tq.timeline,
    save_dir="./",
    noise_low_freq_cutoff = 15.0,
    minimum_segment_duration = 22.0,
    corrupt_trim_length = 0.2,
    max_download_retries = 10,
    retry_delay = 0.5,
    num_workers = 4,
    make_monolithic_file = True,
    sample_rate = 2048.0,
)

drd.download()

2025-12-13 19:14:41 | INFO     | sage.data.download.get_data_release:386 | Downloading segments from H1 for O3a


Validating segments in H1-O3a: 100%|██████████| 6/6 [00:06<00:00,  1.06s/it]

2025-12-13 19:14:47 | INFO     | sage.data.download.get_data_release:396 | H1 O3a: Available = 9535.800000000001, Valid = 9535.800000000001.
Duration and data availability might reduce valid duration.
2025-12-13 19:14:47 | INFO     | sage.data.download.get_data_release:286 | Fetching GWOSC data for detector H1 (O3a) using 4 workers



MP-DET_SCIENCE_DATA GWOSC:   0%|          | 0/3 [00:00<?, ?it/s]

2025-12-13 19:14:51 | ERROR    | sage.dsp.filters:111 | cutoff must be between 0 and Nyquist (fs/2)
2025-12-13 19:14:51 | ERROR    | sage.dsp.filters:111 | cutoff must be between 0 and Nyquist (fs/2)
2025-12-13 19:14:52 | ERROR    | sage.dsp.filters:111 | cutoff must be between 0 and Nyquist (fs/2)


MP-DET_SCIENCE_DATA GWOSC:   0%|          | 0/3 [00:04<?, ?it/s]


ValueError: cutoff must be between 0 and Nyquist (fs/2)

In [ ]:
get_data_release.download_data(
    segments_metadata=tq.timeline,
    save_dir="./",
    noise_low_freq_cutoff = 15.0,
    minimum_segment_duration = 22.0,
    corrupt_trim_length = 0.2,
    max_download_retries = 15,
    retry_delay = 0.5,
    num_workers = 1,
    monolithic_file = True,
    sample_rate = 2048.0,
)

AttributeError: module 'sage.data.download.get_data_release' has no attribute 'download_data'

In [ ]:
import h5py
import numpy as np

In [ ]:
data_1 = h5py.File("data_H1_O3a.h5")
data_2 = h5py.File("data_L1_O3a.h5")
data_3 = h5py.File("data_V1_O3a.h5")

In [ ]:
data_a = h5py.File("./data_H1_O3a/data_H1_O3a.h5")
data_b = h5py.File("./data_H1_O3a/data_L1_O3a.h5")
data_c = h5py.File("./data_H1_O3a/data_V1_O3a.h5")

In [ ]:
data_x = h5py.File("./data_L1_O3a/data_H1_O3a.h5")
data_y = h5py.File("./data_L1_O3a/data_L1_O3a.h5")
data_z = h5py.File("./data_L1_O3a/data_V1_O3a.h5")

In [ ]:
check_1 = np.array(data_1['chunk_0'])
check_2 = np.array(data_a['chunk_0'])
check_3 = np.array(data_x['chunk_0'])

In [ ]:
check_1

array([ -2.24334447,  -3.99765992,   2.95461679, ...,
        14.56597839,   3.4463609 , -10.93714923],
      shape=(9278668,))

In [ ]:
check_2

array([ -7.0655804,  -3.9855285,   5.933519 , ...,  11.474563 ,
        -6.622161 , -14.001852 ],
      shape=(9278668,), dtype=float32)

In [ ]:
check_3

array([ -4.514195 ,  -6.3820868,   0.4610403, ...,  13.246089 ,
         2.1488283, -12.211165 ],
      shape=(9278668,), dtype=float32)